# 00 · Setup check

> **Old question:** Will these tools actually run on *my* laptop?
> **New tool:** one small helper module (`workshop/llm.py`) that talks to any OpenAI-compatible API, plus a tiny decision model running locally.
> **What you'll see:** your settings, one chat answer, one set of probabilities, and the words **Connection OK**.

## How the settings work

All settings live in one file, `config.env`, in the main workshop folder. Copy `config.example.env` to `config.env` and paste your API key after `API_KEY=`. The key stays in that file and never appears in a notebook.

The helpers can run in three **modes**:

| Mode | When | What happens |
| --- | --- | --- |
| **Live** | `API_KEY` is set | New prompts go to the API. Every answer is saved to `outputs_cached/`. |
| **Cached** | the same prompt was answered before | The saved answer is reused instantly with no API call. This works in every mode. |
| **Dry run** | `DRY_RUN=1` | No network at all. Prompts without a saved answer get a **fake** answer that has the right format, so every cell still runs. |

After you edit `config.env`, restart the kernel (**Kernel ▸ Restart Kernel**) so the change is picked up.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # lets this notebook find the workshop/ folder
from workshop import *

status()

## Step 1 · One chat call

`chat()` sends a prompt and returns plain text. In dry-run mode you will see a fake answer marked `[DRY RUN fake answer]`.

In [ ]:
ok = {}
try:
    print(chat("In one short sentence, welcome researchers to a workshop in Beijing."))
    ok["chat"] = True
except Exception as e:
    print("❌ Chat call failed. What to fix:\n", e)
    ok["chat"] = False

## Step 2 · One decision-model call (local)

A *decision model* is a tiny model (about 800 MB) that runs on your laptop through **Ollama**. It does not write text. You give it a piece of text (the *state*) and a question (the *instructions*) with fixed options (the *criteria*), and it returns a **probability for each option**. We use these models in notebook 10.

In [ ]:
question = {"exposure": {"type": "choice",
    "instructions": "How much of this task could a language model do?",
    "criteria": {
    "high": "A language model could do most of this task today",
    "low": "A language model could help with little or none of this task"}}}
try:
    answer = decide("Task: Answer routine customer questions by email.", question)
    print(answer)
    ok["decide"] = True
except Exception as e:
    print("❌ Decision model failed. What to fix:\n", e)
    ok["decide"] = False

In [ ]:
if all(ok.values()):
    print("✅ Connection OK" + ("  (dry run: the answers above are fake)" if mode() == "dry run" else ""))
else:
    print("Fix the ❌ items above, save config.env, then Kernel ▸ Restart Kernel and run all cells again.")

### If something fails

| Message mentions | Fix |
| --- | --- |
| `No API_KEY` | Create `config.env` from `config.example.env` and paste your key. |
| `401` / `key rejected` | The key does not match the `PROVIDER`. Check both lines. |
| `Can't reach` | Check Wi-Fi or VPN. Gemini does not work from mainland China, so use `PROVIDER=minimax_cn`. |
| `Ollama is not running` | Start the Ollama app, then run `ollama pull tev1:0.8b` in a terminal. |
| `not installed` | Run the `ollama pull ...` command printed in the message, or update Ollama from ollama.com. |

The full list is in the README.

## Try it yourself

Change the prompt below and run the cell. Then ask the decision model a yes/no question: replace `"choice"` with `"noul"`, delete the `criteria`, and write a yes/no question in `instructions`. The answer is the probability of "yes".

In [ ]:
print(chat("Explain in two sentences what a codebook is in qualitative research."))